# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code or Rust code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
from LLMClient import get_client, get_models
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
ollama_api_key = os.getenv('OLLAMA_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if ollama_api_key:
    print(f"Anthropic API Key exists and begins {ollama_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")



OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins ad6b7c4
Google API Key exists and begins AQ


In [4]:
# Connect to client libraries
openai = get_client("openai")["client"]
ollama = get_client("ollama")["client"] #this is really ollama
gemini = get_client("google")["client"]

print(openai)
print(ollama)
print(gemini)

In [ ]:
models = ["gemini-2.5-pro", "gemini-2.5-flash" ,"gemini-2.5-flash-lite", "nemotron-3-ultra", "minimax-m3", "gpt-oss:120b", "gemma4:31b", "nemotron-3-nano:30b", "gpt-oss:20b"]

clients = {
            "gemini-2.5-pro": gemini,
            "gemini-2.5-flash": gemini,
            "gemini-2.5-flash-lite": gemini,
            "nemotron-3-ultra": ollama, 
            "gemma4:31b": ollama, 
            "gpt-oss:20b": ollama,
            "minimax-m3": ollama,
            "gpt-oss:120b": ollama
}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [6]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': True,
 'rustc': {'path': '/Users/dpulache/.cargo/bin/rustc',
  'version': 'rustc 1.96.1 (31fca3adb 2026-06-26)',
  'host_triple': 'aarch64-apple-darwin',
  'release': '1.96.1',
  'commit_hash': '31fca3adb283cc9dfd56b49cdee9a96eb9c96ffd'},
 'cargo': {'path': '/Users/dpulache/.cargo/bin/cargo',
  'version': 'cargo 1.96.1 (356927216 2026-06-26)'},
 'rustup': {'path': '/Users/dpulache/.cargo/bin/rustup',
  'version': 'rustup 1.29.0 (28d1352db 2026-03-05)',
  'active_toolchain': 'stable-aarch64-apple-darwin (default)',
  'default_toolchain': '',
  'toolchains': ['stable-aarch64-apple-darwin (active, default)',
   'nightly-aarch64-apple-darwin'],
  'targets_installed': ['aarch64-apple-darwin', 'wasm32-unknown-unknown']},
 'rust_analyzer': {'path': '/Users/dpulache/.cargo/bin/rust-analyzer'},
 'env': {'CARGO_HOME': '/Users/dpulache/.cargo',
  'RUSTUP_HOME': '/Users/dpulache/.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': ['"/Users/dpulache/.cargo

In [10]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = ollama.chat.completions.create(model=models[3], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

You **do not need to install a Rust toolchain**. You already have a stable toolchain (`1.96.1`) and `rustup` installed and configured for `aarch64-apple-darwin`.

Since you want the **fastest possible runtime performance** (compile time is no object) for a single file on an Apple M4, use `rustc` directly with aggressive optimization flags: Full LTO (`fat`), single codegen unit, native CPU tuning, panic abort, and stripped symbols.

```python
import subprocess

# Use the absolute path to your stable rustc for reproducibility
RUSTC = "/Users/dpulache/.cargo/bin/rustc"

compile_command = [
    RUSTC,
    "main.rs",
    "-o", "main",
    # Maximum Optimization
    "-C", "opt-level=3",
    # Full Link Time Optimization (monomorphization across crates; best runtime, slowest compile)
    "-C", "lto=fat",
    # Single codegen unit enables better inter-procedural optimization & inlining
    "-C", "codegen-units=1",
    # Tune specifically for the Apple M4 (ARMv9-A + SVE2, etc.) via LLVM 'native' detection
    "-C", "target-cpu=native",
    # Abort on panic: removes unwinding machinery, reduces binary size, enables more optimization
    "-C", "panic=abort",
    # Strip symbol table & debug info from final binary (smaller, slightly faster load)
    "-C", "strip=symbols",
    "-C", "debuginfo=0",
]

compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

run_command = ["./main"]
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

print(run_result.stdout)
```

### Flag Rationale for M4 / Arm64 macOS
| Flag | Purpose |
| :--- | :--- |
| `lto=fat` | **Fat LTO** merges all MIR into one LLVM module. Best runtime speed; highest memory/compile time. |
| `codegen-units=1` | Forces single LLVM module *before* LTO. Critical for maximal inlining across the whole program. |
| `target-cpu=native` | Instructs LLVM (v19 in Rust 1.96) to detect Apple M4 microarchitecture (FEAT_SVE2, FEAT_I8MM, etc.) and schedule instructions optimally. |
| `panic=abort` | Removes `eh_frame` unwinding tables. Standard for high-perf CLI tools/servers. |
| `strip=symbols` + `debuginfo=0` | Produces minimal binary; faster `execve`/dyld load, better I-cache utilization. |

### Note on `cargo` vs `rustc`
Since `main.rs` is a single file with no `Cargo.toml` dependencies, `rustc` is the correct "simplest" tool. If you later add dependencies, create a `Cargo.toml` and move the flags to `.cargo/config.toml` under `[build] rustflags = [...]`.

## For C++, overwrite this with the commands from yesterday, or for Rust, use the new commands

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [11]:
compile_command = [
    "/Users/dpulache/.cargo/bin/rustc",
    "main.rs",
    "-o", "main",
    # Maximum Optimization
    "-C", "opt-level=3",
    # Full Link Time Optimization (monomorphization across crates; best runtime, slowest compile)
    "-C", "lto=fat",
    # Single codegen unit enables better inter-procedural optimization & inlining
    "-C", "codegen-units=1",
    # Tune specifically for the Apple M4 (ARMv9-A + SVE2, etc.) via LLVM 'native' detection
    "-C", "target-cpu=native",
    # Abort on panic: removes unwinding machinery, reduces binary size, enables more optimization
    "-C", "panic=abort",
    # Strip symbol table & debug info from final binary (smaller, slightly faster load)
    "-C", "strip=symbols",
    "-C", "debuginfo=0",
]

run_command = ["./main"]


## And now, on with the main task

In [12]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [13]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [14]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [15]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [16]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [17]:
# Use the commands from GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [18]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [19]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## RESULTS!

Qwen 2.5 Coder: FAIL  
Gemini 2.5 Pro: FAIL  
DeepSeek Coder v2: FAIL  
Qwen3 Coder 30B: FAIL  
Claude Sonnet 4.5: FAIL    
GPT-5: FAIL    

3rd place: GPT-oss-20B: 0.000341  
2nd place: Grok 4: 0.000317  
**1st place: OpenAI GPT-OSS 120B: 0.000304**  

In [ ]:
print(f"In Ed's experimenet, the GPT-OSS 120B model outcome is {33.755209/0.000304:,.0f} times faster than the Python code.")